# Build & Understand Self-Attention and a Transformer (PyTorch)

##Learning goals
By the end, students will be able to:
* Explain scaled dot-product attention in plain language and with shapes.
* Implement (and read) a minimal Multi-Head Self-Attention (MHSA).
* Stack attention + feed-forward into a tiny Transformer encoder.
* Train a small classifier on a toy sentiment dataset.
* Use masks (padding / causal) and debug common issues.

###**Self-Attention**
Each token
* Asks all tokens a question **(Q)**
* Measures relevance with their keys **(K)**
* Turns those scores into weights (softmax)
* And blends everyone’s information **(V)** into an updated token.

**Scaled dot-product attention:**

$Attn(Q,K,V)=softmax(\frac{QK}{\sqrt(d)})V$

**This shapes:** Shapes (batch $B$, time $T$, model $d$.
$Q,K,V: (B, heads, T, d\_head)→scores(B, heads, T, T)→output(B, heads, T, d_head)→concat heads→(B,T,d)$



###Tokenizer & padding

In [1]:
import math, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

# Tokenizer
PAD, UNK = "<pad>", "<unk>"

def build_vocab(texts, min_freq=1):
    from collections import Counter
    cnt = Counter(w.lower() for t in texts for w in t.split())
    itos = [PAD, UNK] + [w for w,f in cnt.items() if f >= min_freq]
    stoi = {w:i for i,w in enumerate(itos)}
    return stoi, itos

def encode(text, stoi):
    return [stoi.get(w.lower(), stoi[UNK]) for w in text.split()]

def pad_batch(batch, pad_id):
    max_len = max(len(x) for x,_ in batch)
    ids = []
    y   = []
    for x,label in batch:
        ids.append(x + [pad_id]*(max_len-len(x)))
        y.append(label)
    return torch.tensor(ids), torch.tensor(y)


###Scaled dot-product attention (single head)

In [2]:
#Causal LM: set causal=True in attention and train to predict the next word on your tiny corpus.
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_head, dropout=0.1, causal=False):
        super().__init__()
        self.scale = d_head ** 0.5
        self.drop = nn.Dropout(dropout)
        self.causal = causal

    def forward(self, q, k, v, key_mask=None):
        """
        q,k,v: (B, H, T, d_head)
        key_mask: (B, 1, 1, T) with True=keep, False=mask  (optional)
        """
        scores = (q @ k.transpose(-2, -1)) / self.scale  # (B,H,T,T)

        if self.causal:
            T = scores.size(-1)
            causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=scores.device), 1)
            scores = scores.masked_fill(causal, float('-inf'))

        if key_mask is not None:
            scores = scores.masked_fill(~key_mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        attn = self.drop(attn)
        out = attn @ v
        return out, attn


###Multi-Head Self-Attention (MHSA)

In [3]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1, causal=False):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model)
        self.attn = ScaledDotProductAttention(self.d_head, dropout, causal)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        """
        x: (B,T,d_model)
        pad_mask: (B,T) with 1=keep, 0=pad (optional)
        """
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)  # (B,T,D) each

        def split(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)  # (B,H,T,dh)

        q, k, v = map(split, (q, k, v))

        key_mask = None
        if pad_mask is not None:
            key_mask = (pad_mask == 1).unsqueeze(1).unsqueeze(2)  # (B,1,1,T)

        y, _ = self.attn(q, k, v, key_mask=key_mask)             # (B,H,T,dh)
        y = y.transpose(1, 2).contiguous().view(B, T, D)          # (B,T,D)
        y = self.proj_drop(self.proj(y))                          # (B,T,D)
        return y


###Feed-Forward + Transformer block (Pre-LN)

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, causal=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout, causal)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff  = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.attn(self.ln1(x), pad_mask=pad_mask)   # MHSA
        x = x + self.ff(self.ln2(x))                        # FF
        return x


###Tiny Transformer encoder + classifier head

In [5]:
class TinyTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layers=2, n_heads=4, d_ff=256,
                 max_len=256, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout, causal=False)
        for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)  # (1,T)
        x = self.tok(idx) + self.pos(pos)
        x = self.drop(x)
        pad_mask = (idx != self.pad_id).to(torch.bool)

        for blk in self.blocks:
            x = blk(x, pad_mask=pad_mask)
        return self.ln_f(x)  # (B,T,d)

class SentenceClassifier(nn.Module):
    def __init__(self, encoder: TinyTransformerEncoder, n_classes=2, pool="mean"):
        super().__init__()
        self.enc = encoder
        self.pool = pool
        self.head = nn.Linear(self.enc.ln_f.normalized_shape[0], n_classes)

    def forward(self, idx):
        states = self.enc(idx)          # (B,T,d)
        if self.pool == "cls":
            x = states[:, 0]            # take first token
        else:
            mask = (idx != self.enc.pad_id).unsqueeze(-1)  # (B,T,1)
            x = (states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.head(x)             # (B,n_classes)


###Tiny sentiment task (toy dataset)

In [6]:
data = [
    ("I love this movie", 1),
    ("This film is fantastic", 1),
    ("Absolutely wonderful acting", 1),
    ("What a great experience", 1),
    ("It was okay not amazing", 0),
    ("The plot is boring", 0),
    ("I dislike the ending", 0),
    ("Terrible and slow", 0),
    ("Not bad but too long", 0),
    ("A masterpiece with heart", 1),
    ("Mediocre script", 0),
    ("Brilliant direction", 1),
]

texts = [t for t,_ in data]
stoi, itos = build_vocab(texts, min_freq=1)
pad_id = stoi[PAD]

encoded = [(encode(t, stoi), y) for t,y in data]

# split 9/3
train = encoded[:9]
test  = encoded[9:]


###Dataloaders

In [7]:
from torch.utils.data import DataLoader

def collate(batch):  # batch: list of (ids, y)
    return pad_batch(batch, pad_id)

train_loader = DataLoader(train, batch_size=4, shuffle=True, collate_fn=collate)
test_loader  = DataLoader(test,  batch_size=4, shuffle=False, collate_fn=collate)

len_vocab = len(itos)
len_vocab, pad_id

(40, 0)

###Train loop (quick & simple)

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

enc = TinyTransformerEncoder(vocab_size=len_vocab, d_model=64, n_layers=2, n_heads=4,
                             d_ff=128, max_len=64, dropout=0.1, pad_id=pad_id)
model = SentenceClassifier(enc, n_classes=2, pool="mean").to(device)

opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

def run_epoch(loader, train=True):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    for ids, y in loader:
        ids, y = ids.to(device), y.to(device)
        logits = model(ids)
        loss = criterion(logits, y)
        if train:
            opt.zero_grad()
            loss.backward()
            opt.step()
        loss_sum += loss.item() * ids.size(0)
        pred = logits.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += ids.size(0)
    return loss_sum/total, correct/total

for epoch in range(12):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    te_loss, te_acc = run_epoch(test_loader, False)
    if (epoch+1) % 3 == 0:
        print(f"epoch {epoch+1:02d} | train acc={tr_acc:.2f} | test acc={te_acc:.2f}")


epoch 03 | train acc=0.56 | test acc=0.67
epoch 06 | train acc=1.00 | test acc=0.67
epoch 09 | train acc=1.00 | test acc=1.00
epoch 12 | train acc=1.00 | test acc=0.67
